# 24 · Filesystem sandboxing & path traversal

The filesystem server must never read outside its root. `resolve_within_root`
resolves the full path and rejects any escape via `../` or an absolute path.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # PHASE_4_MCP root
import mcp_lab
print('mcp_lab loaded; protocol', mcp_lab.PROTOCOL_VERSION)

In [ ]:
from mcp_lab.safety import resolve_within_root, SandboxError
root = '.'
print('ok  :', resolve_within_root(root, 'mcp_lab/protocol.py').name)
try:
    resolve_within_root(root, '../../../../etc/passwd')
except SandboxError as e:
    print('blocked:', e)

In [ ]:
import sys
from mcp_lab import MiniMCPClient
with MiniMCPClient([sys.executable,'-m','mcp_lab.servers.filesystem']) as c:
    try:
        c.call_tool('read_file', {'path':'../../../../etc/passwd'})
    except RuntimeError as e:
        print('server blocked:', e)

The guard runs *inside* the server, so it holds no matter which client or
agent calls it.